In [1]:
#!pip install -U git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git datasets bitsandbytes peft qwen-vl-utils wandb accelerate
# Tested with transformers==4.53.0.dev0, trl==0.20.0.dev0, datasets==3.6.0, bitsandbytes==0.46.0, peft==0.15.2, qwen-vl-utils==0.0.11, wandb==0.20.1, accelerate==1.8.1

In [2]:
import gc
import time
import torch

def clear_memory():
    # Delete variables if they exist in the current global scope
    if "inputs" in globals():
        del globals()["inputs"]
    if "model" in globals():
        del globals()["model"]
    if "processor" in globals():
        del globals()["processor"]
    if "trainer" in globals():
        del globals()["trainer"]
    if "peft_model" in globals():
        del globals()["peft_model"]
    if "bnb_config" in globals():
        del globals()["bnb_config"]
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(1)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(1)
    gc.collect()
    time.sleep(1)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()

GPU allocated memory: 0.00 GB
GPU reserved memory: 0.00 GB


In [3]:
system_message = """You are a Vision Language Model specialized in detecting clues of fire, smoke and surrounding context then classify them as no fire, dangerous fire or controlled fire.
Your task is to analyze the provided image and respond to queries with concise answers, usually a json format of a caption and a label.
Summarize what you see in the image. Describe the environment, key objects, people, and any signs of fire or smoke.
Based on your summary, classify the fire situation: no fire(e.g., fire alarm, fire distinguisher,..), controlled fire (e.g., fireplace emitting, campfire, cooking, candles, match stick, lighter..) or a dangerous/uncontrolled fire (e.g., curtains on fire, smoke on ceiling, couch on fire, bed sheet on fire, spreading fire on furniture..)
Focus on delivering accurate, succinct caption and precise label based on the visual information. Add a brief explanation for your choice of label in the caption if necessary."""

user_query = """Summarize this situation in the image, look for signs of fire and smoke and classify whether the situation is 
no fire(e.g., fire alarm, fire distinguisher,..), 
controlled fire (e.g., fireplace emitting, campfire, cooking, candles, match stick, lighter..) 
or a dangerous/uncontrolled fire (e.g., curtains on fire, smoke on ceiling, couch on fire, bed sheet on fire, spreading fire on furniture..)
Add a brief explanation for your choice of label in the caption if necessary.
Respond only this json format:
{ \"caption\": \"...\", \"label\": \"no fire\"|\"controlled fire\"|\"dangerous fire\" }
"""

In [4]:
from PIL import Image
import json

def format_data(sample):
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_message}],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "path": f"{sample['image_path']}",
                },
                {
                    "type": "text",
                    "text": "<image>\n"+user_query,
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": f"""```json\n{json.dumps({"caption": sample["caption"], "label": sample["label"]}, ensure_ascii=False)}\n```"""
                }
            ],
        },
    ]


In [5]:
import pandas as pd
# Load the dataset
train_df = pd.read_csv("labels_v2.csv")
val_df = pd.read_csv("val_labels_shortened.csv")
test_df = pd.read_csv("test_labels.csv")

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")


Train size: 452, Val size: 99, Test size: 1290


In [6]:
train_df.head(10)

,image_path,label,caption
0,dataset_v2/image1.jpg,no fire,The image shows a cozy living room with a fire...
1,dataset_v2/image2.jpg,no fire,The image shows a neatly arranged bedroom with...
2,dataset_v2/image3.jpg,no fire,The image shows a dining area with a wooden ta...
3,dataset_v2/image4.jpg,no fire,The image shows a modern office space with lar...
4,dataset_v2/image5.jpg,no fire,The image shows a hospital room with a patient...
5,dataset_v2/image6.jpg,no fire,The image shows a cozy dining area with a roun...
6,dataset_v2/image7.jpg,no fire,The image shows a bedroom with a bed covered i...
7,dataset_v2/image8.jpg,no fire,The image shows an office setting with two com...
8,dataset_v2/image9.jpg,no fire,The image shows a cozy bedroom with a four-pos...
9,dataset_v2/image10.jpg,no fire,The image shows a modern office space with lar...


In [7]:
train_dataset = [format_data(sample) for sample in train_df.to_dict('records')]
eval_dataset = [format_data(sample) for sample in val_df.to_dict('records')]
test_dataset = [format_data(sample) for sample in test_df.to_dict('records')]

In [8]:
train_dataset[:2]

[[{'role': 'system',
   'content': [{'type': 'text',
     'text': 'You are a Vision Language Model specialized in detecting clues of fire, smoke and surrounding context then classify them as no fire, dangerous fire or controlled fire.\nYour task is to analyze the provided image and respond to queries with concise answers, usually a json format of a caption and a label.\nSummarize what you see in the image. Describe the environment, key objects, people, and any signs of fire or smoke.\nBased on your summary, classify the fire situation: no fire(e.g., fire alarm, fire distinguisher,..), controlled fire (e.g., fireplace emitting, campfire, cooking, candles, match stick, lighter..) or a dangerous/uncontrolled fire (e.g., curtains on fire, smoke on ceiling, couch on fire, bed sheet on fire, spreading fire on furniture..)\nFocus on delivering accurate, succinct caption and precise label based on the visual information. Add a brief explanation for your choice of label in the caption if necess

In [9]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch

torch_device = "cuda"
model_checkpoint = "OpenGVLab/InternVL3-2B-hf"
processor = AutoProcessor.from_pretrained(model_checkpoint)
model = AutoModelForImageTextToText.from_pretrained(model_checkpoint, device_map=torch_device, torch_dtype=torch.bfloat16)
print("Successfully loaded the model")

You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Successfully loaded the model


In [10]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/throughput_smolvlm.png"},
                {"type": "text", "text": "<image>\nWhat is this chart about?"},
            ],
        },
    ]

In [11]:
def generate_text_from_sample(model, processor, sample, max_new_tokens=1024, device="cuda"):
    inputs = processor.apply_chat_template(
        sample,
        padding=True,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    generated_ids = model.generate(**inputs, max_new_tokens=1024,pad_token_id=151645)

    trimmed_generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]

    output_text = processor.batch_decode(
        trimmed_generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    return output_text[0]

generate_text_from_sample(model,processor,train_dataset[0])

'```json\n{"caption": "The image shows a cozy living room with a fireplace, a potted plant, a lamp, and a painting on the wall. The room appears to be well-lit and decorated.", "label": "no fire"}\n```'

In [12]:
from peft import LoraConfig, get_peft_model , prepare_model_for_kbit_training

for name, param in model.named_parameters():
    print(name)
    if ("vision_tower" in name or "multi_modal_projector" in name) and param.dtype.is_floating_point:
        param.requires_grad = True

model.vision_tower.embeddings.cls_token
model.vision_tower.embeddings.position_embeddings
model.vision_tower.embeddings.patch_embeddings.projection.weight
model.vision_tower.embeddings.patch_embeddings.projection.bias
model.vision_tower.encoder.layer.0.lambda_1
model.vision_tower.encoder.layer.0.lambda_2
model.vision_tower.encoder.layer.0.attention.q_proj.weight
model.vision_tower.encoder.layer.0.attention.q_proj.bias
model.vision_tower.encoder.layer.0.attention.k_proj.weight
model.vision_tower.encoder.layer.0.attention.k_proj.bias
model.vision_tower.encoder.layer.0.attention.v_proj.weight
model.vision_tower.encoder.layer.0.attention.v_proj.bias
model.vision_tower.encoder.layer.0.attention.projection_layer.weight
model.vision_tower.encoder.layer.0.attention.projection_layer.bias
model.vision_tower.encoder.layer.0.mlp.fc1.weight
model.vision_tower.encoder.layer.0.mlp.fc1.bias
model.vision_tower.encoder.layer.0.mlp.fc2.weight
model.vision_tower.encoder.layer.0.mlp.fc2.bias
model.vision_t

In [13]:
# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="lora_only",
    target_modules=["q_proj", "k_proj", "v_proj", "lora",
                    "up_proj", "down_proj", "gate_proj",
                    "fc1","fc2", 
                    "projection_layer", 
                    "linear_1","linear_2"],          # Vision self-attn output 
    task_type="CAUSAL_LM",
)


# Apply PEFT model adaptation
peft_model = get_peft_model(model, peft_config)

# Print trainable parameters
peft_model.print_trainable_parameters()


trainable params: 12,434,432 || all params: 2,101,110,272 || trainable%: 0.5918


In [14]:
from trl import SFTConfig

# Configure training arguments
training_args = SFTConfig(
    output_dir="internvl3-2B-finetune",  # Directory to save the model
    num_train_epochs=2,  # Number of training epochs
    per_device_train_batch_size=2,  # Batch size for training
    per_device_eval_batch_size=2,  # Batch size for evaluation
    gradient_accumulation_steps=2,  # Steps to accumulate gradients
    gradient_checkpointing=True,  # Enable gradient checkpointing for memory efficiency
    # Optimizer and scheduler settings
    optim="adamw_torch_fused",  # Optimizer type
    learning_rate=1e-5,  # Learning rate for training
    lr_scheduler_type="cosine",  # Type of learning rate scheduler
    # Logging and evaluation
    logging_steps=15,  # Steps interval for logging
    eval_steps=15,  # Steps interval for evaluation
    eval_strategy="steps",  # Strategy for evaluation
    save_strategy="steps",  # Strategy for saving the model
    save_steps=15,  # Steps interval for saving
    metric_for_best_model="eval_loss",  # Metric to evaluate the best model
    greater_is_better=False,  # Whether higher metric values are better
    load_best_model_at_end=True,  # Load the best model after training
    # Mixed precision and gradient settings
    bf16=True,  # Use bfloat16 precision
    max_grad_norm=0.3,  # Maximum norm for gradient clipping
    warmup_ratio=0.1,  # Ratio of total steps for warmup
    # Hub and reporting
    push_to_hub=True,  # Whether to push model to Hugging Face Hub
    report_to="wandb",  # Reporting tool for tracking metrics
    # Gradient checkpointing settings
    gradient_checkpointing_kwargs={"use_reentrant": True},  # Options for gradient checkpointing
    # Dataset configuration
    dataset_kwargs={"skip_prepare_dataset": True},  # Additional dataset options
    # max_seq_length=1024  # Maximum sequence length for input
)

training_args.remove_unused_columns = False  # Keep unused columns in dataset

In [15]:
import wandb

wandb.init(
    project="internvl3-2B-finetune",
    name="internvl3-2B-finetune",
    config=training_args,
)

wandb: Currently logged in as: minhduongqo (minhduongqo-university-of-science-and-technology-of-hano) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [16]:
from PIL import Image

def internvl3_collate_fn(examples):
    # 1️⃣ Build raw text prompts
    texts = [
        processor.apply_chat_template(sample, tokenize=False)
        for sample in examples
    ]

    # 2️⃣ Extract paths and load images now
    images = []
    for sample in examples:
        for msg in sample:
            for c in msg["content"]:
                if c["type"] == "image":
                    img = Image.open(c["path"]).convert("RGB")
                    images.append(img)
                    break

    # 3️⃣ Joint tokenize + image processing
    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    # 4️⃣ Create labels, mask padding
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # 5️⃣ Mask the <image> token in labels
    image_token_id = processor.tokenizer.convert_tokens_to_ids(processor.image_token)
    labels[labels == image_token_id] = -100

    batch["labels"] = labels
    return batch


In [17]:
from trl import SFTTrainer
from torch import cuda

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=internvl3_collate_fn,
    peft_config=peft_config,
    processing_class=processor.tokenizer,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [18]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


/home/student4/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
15,2.628600,2.636503
30,2.525900,2.466234
45,2.338000,2.264826
60,2.137200,2.080786
75,1.953600,1.911857
90,1.803400,1.755581
105,1.653400,1.605874
120,1.506800,1.456476
135,1.365500,1.317789
150,1.243300,1.204029


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.
/home/student4/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/student4/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/student4/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/student4/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/student4/miniconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients w

TrainOutput(global_step=226, training_loss=1.6243216732961943, metrics={'train_runtime': 1351.0268, 'train_samples_per_second': 0.669, 'train_steps_per_second': 0.167, 'total_flos': 1.5732876393449472e+16, 'train_loss': 1.6243216732961943})

In [ ]:
trainer.save_model(training_args.output_dir)

Uploading...:   0%|          | 0.00/60.1M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


: 